## Feed-forward Neural Network

BLEU moyen: 0.023595829946816325

ROUGE-L moyen: 0.09428864264584307

BERTScore moyen: 0.7642185151576996


## Common

In [6]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Flatten, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [7]:
df = pd.read_csv("MovieDataThread.csv")
df = df.dropna(subset=["Script"])
df = df.sample(n=1000, random_state=42)


In [8]:
import re
from nltk.tokenize import word_tokenize

def script_tokenize(text):
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"\n", " NEWLINE_TOKEN ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip().split()

def custom_tokenize(text):
    first_tokenize = script_tokenize(text)
    entokenized = " ".join(first_tokenize)

    second_tokenize = word_tokenize(entokenized)
    return second_tokenize

In [9]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df["Script"].tolist(), test_size=0.2, random_state=42)

In [10]:
train_tokenized = [custom_tokenize(text) for text in train_df]
test_tokenized = [custom_tokenize(text) for text in test_df]

# Train Word2Vec on training data only
w2v_model = Word2Vec(sentences=train_tokenized, vector_size=100, window=5, min_count=2, workers=4)

word_to_idx = {"<PAD>": 0, "<UNK>": 1}
for i, word in enumerate(w2v_model.wv.index_to_key):
    word_to_idx[word] = i + 2
idx_to_word = {i: w for w, i in word_to_idx.items()}

embedding_matrix = np.zeros((len(word_to_idx), 100))
for word, i in word_to_idx.items():
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]

def encode_sequence(tokens):
    return [word_to_idx.get(t, 1) for t in tokens]

def prepare_sequences(tokenized_texts, seq_len=20):
    X, y = [], []
    for tokens in tokenized_texts:
        encoded = encode_sequence(tokens)
        for i in range(len(encoded) - seq_len):
            X.append(encoded[i:i+seq_len])
            y.append(encoded[i+seq_len])
    return np.array(X), np.array(y)

X_train, y_train = prepare_sequences(train_tokenized, seq_len=20)
X_test, y_test = prepare_sequences(test_tokenized, seq_len=20)

X_train, y_train = X_train[:50000], y_train[:50000]



## FFNN Simple

In [ ]:
model = Sequential([
    Embedding(
        input_dim=len(word_to_idx),
        output_dim=100,
        weights=[embedding_matrix],
        input_length=20,
        trainable=True
    ),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(len(word_to_idx), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stopping = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

2025-05-10 13:07:35.329334: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:10:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-05-10 13:07:35.742334: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:10:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-05-10 13:07:35.742401: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:10:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-05-10 13:07:35.744317: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:10:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-05-10 13:07:35.744387: I tensorflow/compile

In [12]:
model.fit(X_train, y_train, validation_split=0.1, batch_size=64, epochs=20, callbacks=[early_stopping])

loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss:.4f} - Test Accuracy: {accuracy:.4f}")

model.save("ffnn_simple_model.h5")

Epoch 1/20


2025-05-10 13:07:58.215084: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x585bce90 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-05-10 13:07:58.215151: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 4060, Compute Capability 8.9
2025-05-10 13:07:58.241645: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-05-10 13:07:58.368880: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:442] Loaded cuDNN version 8902
2025-05-10 13:07:58.522749: I ./tensorflow/compiler/jit/device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


704/704 [==============================] - 32s 38ms/step - loss: 5.6279 - accuracy: 0.1960 - val_loss: 5.2703 - val_accuracy: 0.2220
Epoch 2/20
704/704 [==============================] - 18s 25ms/step - loss: 4.8062 - accuracy: 0.2427 - val_loss: 5.2123 - val_accuracy: 0.2500
Epoch 3/20
704/704 [==============================] - 17s 24ms/step - loss: 4.4438 - accuracy: 0.2615 - val_loss: 5.2696 - val_accuracy: 0.2478
Epoch 4/20
704/704 [==============================] - 18s 25ms/step - loss: 4.1628 - accuracy: 0.2717 - val_loss: 5.2753 - val_accuracy: 0.2554
Epoch 5/20
73249/73249 [==============================] - 1018s 14ms/step - loss: 5.9332 - accuracy: 0.2330
Test Loss: 5.9332 - Test Accuracy: 0.2330


/home/rickgao/miniconda3/envs/tf-wsl/lib/python3.9/site-packages/keras/src/engine/training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [13]:
model.load_weights("ffnn_simple_model.h5")

def generate_text(model, seed_text, word_to_idx, idx_to_word, max_len=50, seq_len=20, temperature=1.0):
    tokens = custom_tokenize(seed_text)
    
    for _ in range(max_len):
        context = tokens[-seq_len:]
        input_indices = [word_to_idx.get(w, word_to_idx["<UNK>"]) for w in context]
        input_indices = [word_to_idx["<PAD>"]] * (seq_len - len(input_indices)) + input_indices
        input_array = np.array(input_indices)[None, :]

        preds = model.predict(input_array, verbose=0)[0]
        preds = np.log(preds + 1e-8) / temperature
        preds = np.exp(preds - np.max(preds))  # avoid overflow
        preds = preds / np.sum(preds)

        next_idx = np.random.choice(len(preds), p=preds)
        next_word = idx_to_word.get(next_idx, "<UNK>")
        if next_word == "<PAD>":
            break
        tokens.append(next_word)

    return " ".join(tokens)


In [14]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge
import re
import string
import math

def calculate_bleu(reference, candidate):
    """
    Calculate BLEU score (it measures the similarity between two sentences)
    :param reference: The reference sentence
    :param candidate: The candidate sentence
    :return: The BLEU score
    """
    smoothing_function = SmoothingFunction().method4
    bleu_score = sentence_bleu([reference], candidate, smoothing_function=smoothing_function)
    return bleu_score

def calculate_rouge(reference, candidate):
    """
    Calculate ROUGE-L score (it calculates the longest common subsequence)
    :param reference: The reference sentence
    :param candidate: The candidate sentence
    :return: The ROUGE score
    """
    rouge = Rouge()
    scores = rouge.get_scores(candidate, reference)
    return scores[0]['rouge-l']['f']

def preprocess_text(text):
    """
    Preprocess the text by removing punctuation and converting to lowercase
    :param text: The input text
    :return: The preprocessed text
    """
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = text.lower()
    return text

def calculate_scores(reference, candidate):
    """
    Calculate BLEU and ROUGE scores for the given reference and candidate sentences
    :param reference: The reference sentence
    :param candidate: The candidate sentence
    :return: The BLEU and ROUGE scores
    """
    # Preprocess the text
    reference = preprocess_text(reference)
    candidate = preprocess_text(candidate)

    reference_tokens = word_tokenize(reference)
    candidate_tokens = word_tokenize(candidate)

    bleu_score = calculate_bleu(reference_tokens, candidate_tokens)
    rouge_score = calculate_rouge(reference, candidate)

    return bleu_score, rouge_score

In [15]:
from bert_score import BERTScorer

scorer = BERTScorer(lang="en", model_type="roberta-base", rescale_with_baseline=True)

bleu_scores, rouge_scores = [], []
bert_f1, bert_precision, bert_recall = [], [], []

PROMPT_LIMIT_TOKEN_SIZE = 25
GENERATE_LIMIT_TOKEN_SIZE = 25

total = len(test_df)
i = 1

for script in test_df:
    print(f"Processing script {i}/{total}")
    script_str = custom_tokenize(script)
    script_str = " ".join(script_str)

    root_token = custom_tokenize(script)
    root_token = " ".join(root_token[:PROMPT_LIMIT_TOKEN_SIZE])

    # Generate text
    generated_text = generate_text(model, root_token, word_to_idx, idx_to_word, max_len=GENERATE_LIMIT_TOKEN_SIZE)
    generated_text = " ".join(generated_text.split()[PROMPT_LIMIT_TOKEN_SIZE:])

    candidate = generated_text
    reference = script_str[len(root_token):len(root_token) + len(candidate)]

    bleu_score, rouge_score = calculate_scores(reference, candidate)

    P, R, F1 = scorer.score([candidate], [reference])
    f1_score = F1.mean().item()
    precision_score = P.mean().item()
    recall_score = R.mean().item()

    bleu_scores.append(bleu_score)
    rouge_scores.append(rouge_score)

    bert_f1.append(f1_score)
    bert_precision.append(precision_score)
    bert_recall.append(recall_score)

    with open(f"ffnn-simple-{PROMPT_LIMIT_TOKEN_SIZE}-{GENERATE_LIMIT_TOKEN_SIZE}.csv", "a") as f:
        f.write(f"{root_token};{candidate};{reference};{bleu_score};{rouge_score};{f1_score};{precision_score};{recall_score}\n")

    i += 1


print("Average BLEU Score :", sum(bleu_scores) / len(bleu_scores))
print("Average ROUGE Score :", sum(rouge_scores) / len(rouge_scores))
print("Average BERT F1 Score :", sum(bert_f1) / len(bert_f1))
print("Average BERT Precision Score :", sum(bert_precision) / len(bert_precision))
print("Average BERT Recall Score :", sum(bert_recall) / len(bert_recall))

/home/rickgao/miniconda3/envs/tf-wsl/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Processing script 1/200
Processing script 2/200
Processing script 3/200
Processing script 4/200
Processing script 5/200
Processing script 6/200
Processing script 7/200
Processing script 8/200
Processing script 9/200
Processing script 10/200
Processing script 11/200
Processing script 12/200
Processing script 13/200
Processing script 14/200
Processing script 15/200
Processing script 16/200
Processing script 17/200
Processing script 18/200
Processing script 19/200
Processing script 20/200
Processing script 21/200
Processing script 22/200
Processing script 23/200
Processing script 24/200
Processing script 25/200
Processing script 26/200
Processing script 27/200
Processing script 28/200
Processing script 29/200
Processing script 30/200
Processing script 31/200
Processing script 32/200
Processing script 33/200
Processing script 34/200
Processing script 35/200
Processing script 36/200
Processing script 37/200
Processing script 38/200
Processing script 39/200
Processing script 40/200
Processin

## FFNN

### Compile

In [16]:
model = Sequential([
    Embedding(
        input_dim=len(word_to_idx),
        output_dim=100,
        weights=[embedding_matrix],
        input_length=20,
        trainable=True
    ),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.4),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(len(word_to_idx), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stopping = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

### Train

In [17]:
model.fit(X_train, y_train, validation_split=0.1, batch_size=64, epochs=20, callbacks=[early_stopping])

loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss:.4f} - Test Accuracy: {accuracy:.4f}")

model.save("fnn_model.h5")

Epoch 1/20
704/704 [==============================] - 26s 35ms/step - loss: 5.8093 - accuracy: 0.1703 - val_loss: 5.4276 - val_accuracy: 0.1976
Epoch 2/20
704/704 [==============================] - 19s 26ms/step - loss: 5.1024 - accuracy: 0.2125 - val_loss: 5.2062 - val_accuracy: 0.2212
Epoch 3/20
704/704 [==============================] - 18s 25ms/step - loss: 4.8069 - accuracy: 0.2297 - val_loss: 5.1083 - val_accuracy: 0.2436
Epoch 4/20
704/704 [==============================] - 18s 26ms/step - loss: 4.6082 - accuracy: 0.2412 - val_loss: 5.0802 - val_accuracy: 0.2478
Epoch 5/20
704/704 [==============================] - 18s 26ms/step - loss: 4.4649 - accuracy: 0.2468 - val_loss: 5.0816 - val_accuracy: 0.2492
Epoch 6/20
704/704 [==============================] - 18s 25ms/step - loss: 4.3498 - accuracy: 0.2560 - val_loss: 5.1094 - val_accuracy: 0.2524
Epoch 7/20
73249/73249 [==============================] - 895s 12ms/step - loss: 5.8250 - accuracy: 0.2317
Test Loss: 5.8250 - Test Accu

/home/rickgao/miniconda3/envs/tf-wsl/lib/python3.9/site-packages/keras/src/engine/training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


### Load

In [18]:
model.load_weights("fnn_model.h5")

def generate_text(model, seed_text, word_to_idx, idx_to_word, max_len=50, seq_len=20, temperature=1.0):
    tokens = custom_tokenize(seed_text)
    
    for _ in range(max_len):
        context = tokens[-seq_len:]
        input_indices = [word_to_idx.get(w, word_to_idx["<UNK>"]) for w in context]
        input_indices = [word_to_idx["<PAD>"]] * (seq_len - len(input_indices)) + input_indices
        input_array = np.array(input_indices)[None, :]

        preds = model.predict(input_array, verbose=0)[0]
        preds = np.log(preds + 1e-8) / temperature
        preds = np.exp(preds - np.max(preds))  # avoid overflow
        preds = preds / np.sum(preds)

        next_idx = np.random.choice(len(preds), p=preds)
        next_word = idx_to_word.get(next_idx, "<UNK>")
        if next_word == "<PAD>":
            break
        tokens.append(next_word)

    return " ".join(tokens)


## Scoring

In [19]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge
import re
import string
import math

def calculate_bleu(reference, candidate):
    """
    Calculate BLEU score (it measures the similarity between two sentences)
    :param reference: The reference sentence
    :param candidate: The candidate sentence
    :return: The BLEU score
    """
    smoothing_function = SmoothingFunction().method4
    bleu_score = sentence_bleu([reference], candidate, smoothing_function=smoothing_function)
    return bleu_score

def calculate_rouge(reference, candidate):
    """
    Calculate ROUGE-L score (it calculates the longest common subsequence)
    :param reference: The reference sentence
    :param candidate: The candidate sentence
    :return: The ROUGE score
    """
    rouge = Rouge()
    scores = rouge.get_scores(candidate, reference)
    return scores[0]['rouge-l']['f']

def preprocess_text(text):
    """
    Preprocess the text by removing punctuation and converting to lowercase
    :param text: The input text
    :return: The preprocessed text
    """
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = text.lower()
    return text

def calculate_scores(reference, candidate):
    """
    Calculate BLEU and ROUGE scores for the given reference and candidate sentences
    :param reference: The reference sentence
    :param candidate: The candidate sentence
    :return: The BLEU and ROUGE scores
    """
    # Preprocess the text
    reference = preprocess_text(reference)
    candidate = preprocess_text(candidate)

    reference_tokens = word_tokenize(reference)
    candidate_tokens = word_tokenize(candidate)

    bleu_score = calculate_bleu(reference_tokens, candidate_tokens)
    rouge_score = calculate_rouge(reference, candidate)

    return bleu_score, rouge_score

## Evaluation

In [20]:
from bert_score import BERTScorer

scorer = BERTScorer(lang="en", model_type="roberta-base", rescale_with_baseline=True)

bleu_scores, rouge_scores = [], []
bert_f1, bert_precision, bert_recall = [], [], []

PROMPT_LIMIT_TOKEN_SIZE = 25
GENERATE_LIMIT_TOKEN_SIZE = 25

total = len(test_df)
i = 1

for script in test_df:
    print(f"Processing script {i}/{total}")
    script_str = custom_tokenize(script)
    script_str = " ".join(script_str)

    root_token = custom_tokenize(script)
    root_token = " ".join(root_token[:PROMPT_LIMIT_TOKEN_SIZE])

    # Generate text
    generated_text = generate_text(model, root_token, word_to_idx, idx_to_word, max_len=GENERATE_LIMIT_TOKEN_SIZE)
    generated_text = " ".join(generated_text.split()[PROMPT_LIMIT_TOKEN_SIZE:])

    candidate = generated_text
    reference = script_str[len(root_token):len(root_token) + len(candidate)]

    bleu_score, rouge_score = calculate_scores(reference, candidate)

    P, R, F1 = scorer.score([candidate], [reference])
    f1_score = F1.mean().item()
    precision_score = P.mean().item()
    recall_score = R.mean().item()

    bleu_scores.append(bleu_score)
    rouge_scores.append(rouge_score)

    bert_f1.append(f1_score)
    bert_precision.append(precision_score)
    bert_recall.append(recall_score)

    with open(f"ffnn-{PROMPT_LIMIT_TOKEN_SIZE}-{GENERATE_LIMIT_TOKEN_SIZE}.csv", "a") as f:
        f.write(f"{root_token};{candidate};{reference};{bleu_score};{rouge_score};{f1_score};{precision_score};{recall_score}\n")

    i += 1


print("Average BLEU Score :", sum(bleu_scores) / len(bleu_scores))
print("Average ROUGE Score :", sum(rouge_scores) / len(rouge_scores))
print("Average BERT F1 Score :", sum(bert_f1) / len(bert_f1))
print("Average BERT Precision Score :", sum(bert_precision) / len(bert_precision))
print("Average BERT Recall Score :", sum(bert_recall) / len(bert_recall))

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Processing script 1/200
Processing script 2/200
Processing script 3/200
Processing script 4/200
Processing script 5/200
Processing script 6/200
Processing script 7/200
Processing script 8/200
Processing script 9/200
Processing script 10/200
Processing script 11/200
Processing script 12/200
Processing script 13/200
Processing script 14/200
Processing script 15/200
Processing script 16/200
Processing script 17/200
Processing script 18/200
Processing script 19/200
Processing script 20/200
Processing script 21/200
Processing script 22/200
Processing script 23/200
Processing script 24/200
Processing script 25/200
Processing script 26/200
Processing script 27/200
Processing script 28/200
Processing script 29/200
Processing script 30/200
Processing script 31/200
Processing script 32/200
Processing script 33/200
Processing script 34/200
Processing script 35/200
Processing script 36/200
Processing script 37/200
Processing script 38/200
Processing script 39/200
Processing script 40/200
Processin

## FFNN Normalize

### Compile

In [21]:
from tensorflow.keras.layers import LayerNormalization

model = Sequential([
    Embedding(
        input_dim=len(word_to_idx),
        output_dim=100,
        weights=[embedding_matrix],
        input_length=20,
        trainable=True
    ),
    Flatten(),
    Dense(256, activation='relu'),
    LayerNormalization(),
    Dropout(0.4),
    Dense(128, activation='relu'),
    LayerNormalization(),
    Dropout(0.3),
    Dense(len(word_to_idx), activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stopping = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

### Train

In [22]:
model.fit(X_train, y_train, validation_split=0.1, batch_size=64, epochs=20, callbacks=[early_stopping])

loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss:.4f} - Test Accuracy: {accuracy:.4f}")

model.save("fnn_normalize_model.h5")

Epoch 1/20
704/704 [==============================] - 31s 41ms/step - loss: 5.6357 - accuracy: 0.1784 - val_loss: 5.0817 - val_accuracy: 0.2250
Epoch 2/20
704/704 [==============================] - 22s 31ms/step - loss: 4.5948 - accuracy: 0.2455 - val_loss: 4.8211 - val_accuracy: 0.2504
Epoch 3/20
704/704 [==============================] - 21s 30ms/step - loss: 4.2599 - accuracy: 0.2701 - val_loss: 4.7261 - val_accuracy: 0.2622
Epoch 4/20
704/704 [==============================] - 20s 29ms/step - loss: 4.0404 - accuracy: 0.2826 - val_loss: 4.7517 - val_accuracy: 0.2686
Epoch 5/20
704/704 [==============================] - 18s 26ms/step - loss: 3.8651 - accuracy: 0.2960 - val_loss: 4.8201 - val_accuracy: 0.2712
Epoch 6/20
73249/73249 [==============================] - 1059s 14ms/step - loss: 5.3157 - accuracy: 0.2472
Test Loss: 5.3157 - Test Accuracy: 0.2472


/home/rickgao/miniconda3/envs/tf-wsl/lib/python3.9/site-packages/keras/src/engine/training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


### Load

In [23]:
model.load_weights("fnn_normalize_model.h5")

def generate_text(model, seed_text, word_to_idx, idx_to_word, max_len=50, seq_len=20, temperature=1.0):
    tokens = custom_tokenize(seed_text)
    
    for _ in range(max_len):
        context = tokens[-seq_len:]
        input_indices = [word_to_idx.get(w, word_to_idx["<UNK>"]) for w in context]
        input_indices = [word_to_idx["<PAD>"]] * (seq_len - len(input_indices)) + input_indices
        input_array = np.array(input_indices)[None, :]

        preds = model.predict(input_array, verbose=0)[0]
        preds = np.log(preds + 1e-8) / temperature
        preds = np.exp(preds - np.max(preds))  # avoid overflow
        preds = preds / np.sum(preds)

        next_idx = np.random.choice(len(preds), p=preds)
        next_word = idx_to_word.get(next_idx, "<UNK>")
        if next_word == "<PAD>":
            break
        tokens.append(next_word)

    return " ".join(tokens)


## Scoring

In [24]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge
import re
import string
import math

def calculate_bleu(reference, candidate):
    """
    Calculate BLEU score (it measures the similarity between two sentences)
    :param reference: The reference sentence
    :param candidate: The candidate sentence
    :return: The BLEU score
    """
    smoothing_function = SmoothingFunction().method4
    bleu_score = sentence_bleu([reference], candidate, smoothing_function=smoothing_function)
    return bleu_score

def calculate_rouge(reference, candidate):
    """
    Calculate ROUGE-L score (it calculates the longest common subsequence)
    :param reference: The reference sentence
    :param candidate: The candidate sentence
    :return: The ROUGE score
    """
    rouge = Rouge()
    scores = rouge.get_scores(candidate, reference)
    return scores[0]['rouge-l']['f']

def preprocess_text(text):
    """
    Preprocess the text by removing punctuation and converting to lowercase
    :param text: The input text
    :return: The preprocessed text
    """
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = text.lower()
    return text

def calculate_scores(reference, candidate):
    """
    Calculate BLEU and ROUGE scores for the given reference and candidate sentences
    :param reference: The reference sentence
    :param candidate: The candidate sentence
    :return: The BLEU and ROUGE scores
    """
    # Preprocess the text
    reference = preprocess_text(reference)
    candidate = preprocess_text(candidate)

    reference_tokens = word_tokenize(reference)
    candidate_tokens = word_tokenize(candidate)

    bleu_score = calculate_bleu(reference_tokens, candidate_tokens)
    rouge_score = calculate_rouge(reference, candidate)

    return bleu_score, rouge_score

## Evaluation

In [25]:
from bert_score import BERTScorer

scorer = BERTScorer(lang="en", model_type="roberta-base", rescale_with_baseline=True)

bleu_scores, rouge_scores = [], []
bert_f1, bert_precision, bert_recall = [], [], []

PROMPT_LIMIT_TOKEN_SIZE = 25
GENERATE_LIMIT_TOKEN_SIZE = 25

total = len(test_df)
i = 1

for script in test_df:
    print(f"Processing script {i}/{total}")
    script_str = custom_tokenize(script)
    script_str = " ".join(script_str)

    root_token = custom_tokenize(script)
    root_token = " ".join(root_token[:PROMPT_LIMIT_TOKEN_SIZE])

    # Generate text
    generated_text = generate_text(model, root_token, word_to_idx, idx_to_word, max_len=GENERATE_LIMIT_TOKEN_SIZE)
    generated_text = " ".join(generated_text.split()[PROMPT_LIMIT_TOKEN_SIZE:])

    candidate = generated_text
    reference = script_str[len(root_token):len(root_token) + len(candidate)]

    bleu_score, rouge_score = calculate_scores(reference, candidate)

    P, R, F1 = scorer.score([candidate], [reference])
    f1_score = F1.mean().item()
    precision_score = P.mean().item()
    recall_score = R.mean().item()

    bleu_scores.append(bleu_score)
    rouge_scores.append(rouge_score)

    bert_f1.append(f1_score)
    bert_precision.append(precision_score)
    bert_recall.append(recall_score)

    with open(f"ffnn-{PROMPT_LIMIT_TOKEN_SIZE}-{GENERATE_LIMIT_TOKEN_SIZE}.csv", "a") as f:
        f.write(f"{root_token};{candidate};{reference};{bleu_score};{rouge_score};{f1_score};{precision_score};{recall_score}\n")

    i += 1


print("Average BLEU Score :", sum(bleu_scores) / len(bleu_scores))
print("Average ROUGE Score :", sum(rouge_scores) / len(rouge_scores))
print("Average BERT F1 Score :", sum(bert_f1) / len(bert_f1))
print("Average BERT Precision Score :", sum(bert_precision) / len(bert_precision))
print("Average BERT Recall Score :", sum(bert_recall) / len(bert_recall))

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Processing script 1/200
Processing script 2/200
Processing script 3/200
Processing script 4/200
Processing script 5/200
Processing script 6/200
Processing script 7/200
Processing script 8/200
Processing script 9/200
Processing script 10/200
Processing script 11/200
Processing script 12/200
Processing script 13/200
Processing script 14/200
Processing script 15/200
Processing script 16/200
Processing script 17/200
Processing script 18/200
Processing script 19/200
Processing script 20/200
Processing script 21/200
Processing script 22/200
Processing script 23/200
Processing script 24/200
Processing script 25/200
Processing script 26/200
Processing script 27/200
Processing script 28/200
Processing script 29/200
Processing script 30/200
Processing script 31/200
Processing script 32/200
Processing script 33/200
Processing script 34/200
Processing script 35/200
Processing script 36/200
Processing script 37/200
Processing script 38/200
Processing script 39/200
Processing script 40/200
Processin